#### Mixed Variables Detection & Cleaning

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# ─── Inline Dataset: Ola Ride Records ───
data = {
    'ride_id':   [101, 102, 103, 104, 105, 106, 107, 108],
    'distance':  ['3.2 km', '5 km', 'N/A', '8.1 km',        # Number + Unit + Special
                  'Cancelled', '12.5 km', '2 km', '6.8 km'],
    'fare':      ['120', '200', 'Pending', '310',             # Number + String
                  'Refunded', '450', '90', '250'],
    'product_code': ['A100', 'B200', 'A150', 'C300',         # Code + Category mix
                     'B180', 'A200', 'C100', 'B250'],
}
df = pd.DataFrame(data)

print("Original Data:")
print(df)
print("\nData Types:")
print(df.dtypes)

In [ ]:
# ════════════════════════════════════════
# FIX 1: distance column — unit hatao, number nikalo
# Approach: str.replace() se ' km' hatao, phir pd.to_numeric se convert karo
# ════════════════════════════════════════
# Step 1: ' km' text hatao
df['distance_clean'] = df['distance'].str.replace(' km', '', case=False)
# Step 2: Special words (N/A, Cancelled) ko NaN kar do
special_vals = ['N/A', 'Cancelled', 'Unknown', '']
df['distance_clean'] = df['distance_clean'].apply(
    lambda x: np.nan if x in special_vals else x
)
# Step 3: pd.to_numeric — jo bhi valid number hai woh float banega, baaki NaN
df['distance_km'] = pd.to_numeric(df['distance_clean'], errors='coerce')

df[['distance', 'distance_km']]

In [ ]:
# ════════════════════════════════════════
# FIX 2: fare column — numeric convert karo
# pd.to_numeric se: '120' → 120.0 | 'Pending' → NaN (errors='coerce')
# ════════════════════════════════════════
df['fare_num'] = pd.to_numeric(df['fare'], errors='coerce')
# 'Pending', 'Refunded' automatically NaN ho jaate hain!

df[['fare', 'fare_num']]

In [ ]:
# ════════════════════════════════════════
# FIX 3: product_code — category aur number alag karo
# 'A100' → category='A', number=100
# str[0] = pehla character (letter), str[1:] = baaki (number)
# ════════════════════════════════════════
df['product_category'] = df['product_code'].str[0]       # 'A100' → 'A'
df['product_num']      = pd.to_numeric(df['product_code'].str[1:])  # 'A100' → 100

df[['product_code', 'product_category', 'product_num']]

In [ ]:
# ════════════════════════════════════════
# FINAL RESULT
# ════════════════════════════════════════
# Temporary column drop karo
df.drop(columns=['distance_clean'], inplace=True)

print("\nNew Data Types:")
print(df[['distance_km','fare_num','product_num']].dtypes)

print("\nCleaned Dataset:")
df[['ride_id','distance_km','fare_num','product_category','product_num']]

#### Date & Time Features Extraction

In [ ]:
# ─── Inline Dataset: Swiggy Orders ───
data = {
    'order_id':  [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    'order_time': ['2024-01-15 13:30:00', '2024-03-22 20:15:00',
                   '2024-06-10 08:45:00', '2024-10-25 19:00:00',
                   '2024-12-25 12:00:00', '2024-04-07 22:30:00',
                   '2024-08-15 07:15:00', '2024-11-01 15:45:00'],
    'city':      ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad',
                  'Chennai', 'Pune', 'Kolkata', 'Ahmedabad'],
    'amount':    [350, 520, 180, 430, 850, 290, 620, 410],
}

df = pd.DataFrame(data)

df

In [ ]:
# ════════════════════════════════════════
# STEP 1: String → datetime convert karo
# ════════════════════════════════════════
df['order_time'] = pd.to_datetime(df['order_time'])
print("Data type after conversion:", df['order_time'].dtype)
# datetime64[ns]

In [ ]:
# ════════════════════════════════════════
# STEP 2: Basic date features nikalo
# ════════════════════════════════════════
df['year']     = df['order_time'].dt.year        # 2024
df['month']    = df['order_time'].dt.month       # 1-12
df['day']      = df['order_time'].dt.day         # 1-31
df['quarter']  = df['order_time'].dt.quarter     # 1-4
df['weekday']  = df['order_time'].dt.dayofweek   # 0=Mon, 6=Sun
df['week_num'] = df['order_time'].dt.isocalendar().week.astype(int)

# ════════════════════════════════════════
# STEP 3: Time features nikalo
# ════════════════════════════════════════
df['hour']   = df['order_time'].dt.hour    # 0-23
df['minute'] = df['order_time'].dt.minute  # 0-59


df[['order_time', 'year', 'month', 'day', 'hour', 'minute', 'quarter', 'weekday', 'week_num']]

In [ ]:
# ════════════════════════════════════════
# STEP 4: Derived / Business features banao
# ════════════════════════════════════════
# Weekend flag
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)  # 5=Sat, 6=Sun

# Time of day category
def time_of_day(hour):
    if 5 <= hour < 12:   return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else:                 return 'Night'

df['time_of_day'] = df['hour'].apply(time_of_day)

# Indian Season (roughly)
def indian_season(month):
    if month in [3, 4, 5]:   return 'Summer'
    elif month in [6, 7, 8, 9]: return 'Monsoon'
    elif month in [10, 11]:  return 'Autumn'
    else:                    return 'Winter'  # 12, 1, 2

df['season'] = df['month'].apply(indian_season)

# Days since a reference date
reference_date = pd.Timestamp('2024-01-01')
df['days_since_newyear'] = (df['order_time'] - reference_date).dt.days

# ════════════════════════════════════════
# RESULT
# ════════════════════════════════════════
print("\nExtracted Features:")

df[['order_id','month','weekday','hour','is_weekend', 'time_of_day','season','days_since_newyear']]

#### Complete Case Analysis (CCA)

In [ ]:
# ─── Inline Dataset: Student Exam Records ───
data = {
    'student':  ['Aarav','Priya','Ravi','Sneha','Karan',
                 'Divya','Mohit','Ananya','Vikram','Pooja'],
    'age':      [22, np.nan, 25, 23, np.nan, 21, 24, np.nan, 26, 20],
    'marks':    [85, 90, np.nan, 78, 88, np.nan, 72, 95, 80, np.nan],
    'city':     ['Mumbai','Delhi', np.nan,'Surat','Mumbai',
                 np.nan,'Delhi','Pune','Bangalore','Chennai'],
    'attendance':[90, 85, 80, np.nan, 92, 88, np.nan, 95, 78, 85],
}

df = pd.DataFrame(data)

print("Original Dataset:")
print(f"\nShape: {df.shape}")
print(f"\nMissing values per column:\n{df.isnull().sum()}")
print(f"\nRows with at least one NaN: {df.isnull().any(axis=1).sum()}")
df

In [ ]:
# ════════════════════════════════════════
# METHOD 1: All NaN rows hatao (basic CCA)
# ════════════════════════════════════════
df_cca = df.dropna()

print(f"\n--- Basic CCA (dropna all) ---")
print(f"Rows before: {len(df)} | Rows after: {len(df_cca)}")

df_cca

In [ ]:
# ════════════════════════════════════════
# METHOD 2: Specific columns pe CCA
# sirf marks aur age missing ho toh hi drop karo
# ════════════════════════════════════════
df_cca2 = df.dropna(subset=['marks', 'age'])

print(f"\n--- CCA on specific columns (marks + age) ---")
print(f"Rows before: {len(df)} | Rows after: {len(df_cca2)}")

df_cca2

In [ ]:
# ════════════════════════════════════════
# METHOD 3: Threshold — agar 2+ columns NaN hain toh drop
# ════════════════════════════════════════
# thresh=3 means: kam se kam 4 non-NaN values chahiye row mein
df_cca4 = df.dropna(thresh=4)
print(f"\n--- CCA with thresh=4 ---")
print(f"Rows before: {len(df)} | Rows after: {len(df_cca4)}")

df_cca4

In [ ]:
# ════════════════════════════════════════
# BEFORE vs AFTER — Statistics Check
# ════════════════════════════════════════
print("\n--- Before vs After: marks column ---")
print(f"Mean BEFORE: {df['marks'].mean():.2f}")
print(f"Mean AFTER:  {df_cca['marks'].mean():.2f}")
print(f"Missing %:   {df.isnull().sum().sum() / df.size * 100:.1f}% original")
print(f"Missing %:   {df_cca.isnull().sum().sum() / df_cca.size * 100:.1f}% after CCA")

#### Complete Data Cleaning Pipeline — Mixed + DateTime + CCA

In [ ]:
# ─── Inline Dataset: Ola Rides with all problem types ───
data = {
    'ride_id':   [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'booking_time': ['2024-03-15 08:30:00','2024-06-22 14:15:00',
                     '2024-10-05 20:00:00','2024-01-18 07:45:00',
                     None,                 '2024-08-11 18:30:00',
                     '2024-12-25 11:00:00','2024-04-30 16:20:00',
                     '2024-09-01 09:10:00','2024-07-14 21:45:00'],
    'distance':  ['5.2 km','8 km','N/A','12.5 km','3.8 km',
                  'Cancelled','6.1 km','9.3 km',None,'4.7 km'],
    'fare':      ['180','250','Pending','420','140',
                  'Refunded','210','310',None,'165'],
    'rating':    [4.5, None, 4.2, 4.8, 4.0, None, 4.7, 4.3, 4.6, None],
    'city':      ['Mumbai','Delhi',None,'Pune','Bangalore',
                  'Mumbai','Chennai',None,'Hyderabad','Kolkata'],
}
df = pd.DataFrame(data)

print("=== STEP 1: Original Dataset ===")
print(f"\nShape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")

df

In [ ]:
# ════════════════════════════════
# STEP 2: Date-Time Handle karo
# ════════════════════════════════
df['booking_time'] = pd.to_datetime(df['booking_time'], errors='coerce')
df['hour']         = df['booking_time'].dt.hour
df['month']        = df['booking_time'].dt.month
df['is_weekend']   = df['booking_time'].dt.dayofweek.isin([5,6]).astype('Int64')

print("\n=== STEP 2: DateTime Features Added ===")

df[['ride_id','hour','month','is_weekend']]

In [ ]:
# ════════════════════════════════
# STEP 3: Mixed Variables Clean karo
# Simple approach: str.replace + pd.to_numeric
# ════════════════════════════════

# distance: ' km' text hatao, special values NaN, phir numeric convert
special_vals = ['N/A', 'Cancelled', 'Refunded', 'Pending', 'Unknown', '']

df['distance_clean'] = df['distance'].str.replace(' km', '', case=False)
df['distance_clean'] = df['distance_clean'].apply(
    lambda x: np.nan if x in special_vals else x
)
df['distance_km'] = pd.to_numeric(df['distance_clean'], errors='coerce')

# fare: sirf pd.to_numeric — 'Pending', 'Refunded' automatically NaN ho jaate
df['fare_num'] = pd.to_numeric(df['fare'], errors='coerce')

# Cleanup temporary column
df.drop(columns=['distance_clean'], inplace=True)

print("\n=== STEP 3: Mixed Variables Cleaned ===")

df[['ride_id','distance_km','fare_num']]

In [ ]:
# ════════════════════════════════
# STEP 4: CCA — Complete Case Analysis
# sirf core columns pe apply karo
# ════════════════════════════════
core_cols = ['distance_km', 'fare_num', 'rating', 'city', 'hour']
df_clean = df.dropna(subset=core_cols)

print(f"\n=== STEP 4: CCA Applied ===")
print(f"Rows before CCA: {len(df)}")
print(f"Rows after  CCA: {len(df_clean)}")
print(f"\nFinal Clean Dataset:")

df_clean[['ride_id','distance_km','fare_num',
                'rating','city','hour','month','is_weekend']]